# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `book/chapters/ch05_data_preprocessing.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 데이터를 무조건 깨끗하게 만드는 것이 아니라, 분석 결과를 신뢰할 수 있도록 데이터의 상태를 확인하고 처리 기준을 남기는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/`에 그대로 둡니다.
- 전처리 결과는 `data/processed/`에 별도로 저장합니다.
- 전처리 요약 보고서는 `reports/ch05_preprocessing_summary.md`에 저장합니다.
- 코드가 실행되었다고 끝이 아니라, 처리 기준을 설명할 수 있어야 합니다.


## 1. 왜 전처리가 필요한가

현실의 데이터는 처음부터 분석하기 좋은 형태로 주어지지 않습니다.

예를 들어 다음과 같은 문제가 있을 수 있습니다.

| 문제 유형 | 예시 | 확인할 질문 |
|---|---|---|
| 결측치 | 나이가 비어 있는 고객 | 비어 있는 이유가 무엇인가? 삭제해도 되는가? |
| 중복 | 같은 고객 ID가 여러 번 등장 | 자연스러운 중복인가, 데이터 오류인가? |
| 타입 오류 | 가격이 문자열로 저장됨 | 계산 가능한 숫자형으로 바꿀 수 있는가? |
| 날짜 오류 | 주문일이 문자열로 저장됨 | 월별·요일별 분석에 사용할 수 있는가? |
| 문자열 표기 차이 | `Seoul`, ` Seoul`, `SEOUL` | 같은 값을 하나의 표기로 통일해야 하는가? |
| 이상값 | 수량이 음수이거나 가격이 0원 | 실제 의미가 있는 값인가, 입력 오류인가? |
| 파생 컬럼 필요 | 수량과 단가만 있고 주문금액이 없음 | 분석에 필요한 새 컬럼을 만들 수 있는가? |


## 2. 원본 데이터와 전처리 데이터 분리하기

전처리에서 가장 중요한 원칙은 원본 데이터를 직접 덮어쓰지 않는 것입니다.

```text
data/
├─ raw/
│  ├─ customers.csv
│  ├─ products.csv
│  ├─ orders.csv
│  └─ order_items.csv
└─ processed/
   ├─ customers_clean.csv
   ├─ products_clean.csv
   ├─ orders_clean.csv
   └─ order_items_clean.csv
```

원본과 전처리 결과를 분리하면 실수했을 때 원본으로 돌아갈 수 있고, 전처리 전후 차이를 비교할 수 있습니다.


## 3. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: c:\dev\llm-data-analysis-course\notebooks
프로젝트 루트: c:\dev\llm-data-analysis-course
원본 데이터 폴더: c:\dev\llm-data-analysis-course\data\raw
전처리 데이터 폴더: c:\dev\llm-data-analysis-course\data\processed
보고서 폴더: c:\dev\llm-data-analysis-course\reports


## 4. 원본 데이터 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보


In [2]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')


원본 데이터 불러오기 완료


## 5. 전처리 전 데이터 크기 기록하기

전처리 전 데이터 크기를 기록해 두면 나중에 어떤 데이터가 얼마나 바뀌었는지 비교할 수 있습니다.


In [3]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes


,dataset,rows,columns
0,customers,150,6
1,products,100,4
2,orders,300,5
3,order_items,764,5


In [4]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (764, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



## 6. 원본을 복사해서 전처리 시작하기

전처리 과정에서는 원본 DataFrame을 직접 수정하지 않고 복사본을 사용합니다.


In [5]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')


전처리용 복사본 생성 완료


## 7. 결측치 확인하기

결측치를 처리하기 전에 먼저 어디에 얼마나 비어 있는 값이 있는지 확인합니다. 개수와 비율을 함께 보면 데이터 크기에 따른 차이를 이해하기 쉽습니다.


In [6]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())



[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [7]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

missing_summary(customers_clean)


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 8. 결측치 처리하기

결측치 처리는 컬럼의 의미에 따라 달라집니다.

- 나이처럼 숫자형 컬럼은 중앙값으로 대체할 수 있습니다.
- 도시처럼 범주형 컬럼은 `Unknown`으로 남겨 분석에서 구분할 수 있습니다.

실제 업무에서는 결측치가 왜 생겼는지 먼저 확인해야 합니다.


In [8]:
if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    age_median = customers_clean['age'].median()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

missing_summary(customers_clean)


age 중앙값: 40.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 9. 중복 확인하기

중복은 맥락에 따라 의미가 다릅니다. `customers`의 `customer_id` 중복은 오류일 가능성이 크지만, `order_items`의 `order_id` 반복은 한 주문에 여러 상품이 들어갈 수 있으므로 자연스럽습니다.


In [9]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())


customers 전체 행 중복 수: 0
products 전체 행 중복 수: 0
orders 전체 행 중복 수: 0
order_items 전체 행 중복 수: 0


In [10]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')


customers customer_id 중복 수: 0
products product_id 중복 수: 0
orders order_id 중복 수: 0
order_items order_item_id 중복 수: 0


In [11]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')


완전 중복 행 제거 완료


## 10. 문자열 표기 정리하기

문자열 데이터에는 앞뒤 공백이 숨어 있을 수 있습니다. `Seoul`과 ` Seoul `은 눈으로는 비슷하지만 pandas에서는 서로 다른 값입니다.

결측치를 문자열 `nan`으로 바꾸지 않도록 주의하면서 문자열 컬럼의 앞뒤 공백을 제거합니다.


In [12]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')


문자열 앞뒤 공백 제거 완료


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_8488\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_8488\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pan

In [13]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 11. 주문 상태값 통일하기

주문 상태값이 `complete`, `Complete`, `COMPLETED`, `완료`처럼 섞여 있으면 같은 상태를 서로 다른 값으로 집계할 수 있습니다. 대표 표기로 통일합니다.


In [14]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 12. 날짜 컬럼 변환하기

날짜처럼 보이는 값도 실제로는 문자열일 수 있습니다. 월별 주문 분석이나 요일별 분석을 하려면 날짜형으로 변환해야 합니다.

`errors="coerce"`는 변환할 수 없는 값을 `NaT`로 바꿉니다. 그래서 변환 후 실패 건수를 반드시 확인해야 합니다.


In [15]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()


order_date 변환 실패: 0
주문 시작일: 2025-09-14 00:00:00
주문 종료일: 2026-09-13 00:00:00


,order_date,order_month,order_dayofweek
0,2026-07-13,2026-07,Monday
1,2025-09-28,2025-09,Sunday
2,2026-01-25,2026-01,Sunday
3,2026-04-07,2026-04,Tuesday
4,2026-02-26,2026-02,Thursday


In [16]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())


signup_date 변환 실패: 0


,customer_id,signup_date
0,1,2024-08-25
1,2,2026-01-08
2,3,2024-08-18
3,4,2026-06-19
4,5,2024-11-19


## 13. 숫자형 컬럼 변환하기

가격, 수량, 단가는 숫자처럼 보여도 문자열일 수 있습니다. 특히 `10,000`처럼 쉼표가 들어간 값은 바로 계산하기 어렵습니다.


In [17]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())


price 변환 실패: 0
quantity 변환 실패: 0
unit_price 변환 실패: 0


## 14. 이상값 후보 확인하기

이상값은 일반적인 범위를 벗어난 값입니다. 하지만 이상값이 항상 오류는 아닙니다. 0원 상품은 이벤트 상품일 수 있고, 음수 수량은 반품을 의미할 수도 있습니다. 먼저 확인하고 기준을 정해야 합니다.


In [18]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())


count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

,quantity,unit_price
count,764.000000,764.000000
mean,3.053665,108561.518325
std,1.410873,56996.770604
min,1.000000,5000.000000
25%,2.000000,62000.000000
50%,3.000000,111000.000000
75%,4.000000,161250.000000
max,5.000000,200000.000000


In [19]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))


price <= 0: 0
quantity <= 0: 0
unit_price <= 0: 0


### 실습용 이상값 처리 기준

이번 실습에서는 정상 주문 분석을 위해 다음 기준을 적용합니다.

```text
- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외
```

실제 업무에서는 이 기준을 적용하기 전에 반드시 원본 시스템이나 담당자 확인이 필요합니다.


In [20]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)


이상값 처리 후 products: (100, 4)
이상값 처리 후 order_items: (764, 5)


## 15. 파생 컬럼 만들기

전처리된 수량과 단가를 사용하면 주문 상세 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [21]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,3,102000,306000
1,5,25000,125000
2,3,142000,426000
3,3,193000,579000
4,4,189000,756000


In [22]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())


전처리 후 주문 상세 금액 합계: 255610000


## 16. 파일 간 관계 다시 확인하기

전처리 과정에서 일부 행을 삭제하면 파일 간 관계가 깨질 수 있습니다. 예를 들어 `products`에서 일부 상품을 제외했는데 `order_items`에는 그 상품 ID가 남아 있을 수 있습니다.


In [23]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


## 17. 전처리 전후 비교하기

전처리 전후 데이터 크기를 비교합니다. 행 수가 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 하고, 열 수가 늘었다면 어떤 파생 컬럼이 추가되었는지 설명할 수 있어야 합니다.


In [24]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,products,100,4,100,4
2,orders,300,5,300,7
3,order_items,764,5,764,6


## 18. 전처리 결과 저장하기

전처리된 데이터는 `data/processed` 폴더에 저장합니다. Excel에서 한글 CSV를 바로 열 수 있도록 `utf-8-sig` 인코딩을 사용합니다.


In [25]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

list(PROCESSED_DIR.glob('*_clean.csv'))


[WindowsPath('c:/dev/llm-data-analysis-course/data/processed/customers_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/data/processed/orders_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/data/processed/order_items_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/data/processed/products_clean.csv')]

## 19. 전처리 요약 보고서 저장하기

전처리 기준과 결과를 Markdown 파일로 남겨 두면 이후 보고서 작성이 쉬워집니다.


In [26]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)


요약 보고서 저장 완료: c:\dev\llm-data-analysis-course\reports\ch05_preprocessing_summary.md


## 20. 전처리 과정을 함수로 정리하기

전처리 코드를 한 번만 실행하고 끝내면 재사용하기 어렵습니다. 같은 데이터를 다시 받거나 처리 기준을 조금 바꾸어야 할 때는 함수로 정리된 코드가 훨씬 유용합니다.


In [27]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    return result.drop_duplicates()


In [28]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)


(150, 6)
(100, 4)
(300, 7)
(764, 6)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_8488\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_8488\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pan

## 21. 소스 모듈 사용하기

위에서 직접 작성한 전처리 함수는 `src/preprocessing.py`에도 정리되어 있습니다. 반복 작업이나 실제 프로젝트에서는 노트북 안에만 코드를 두기보다 소스 모듈로 분리하는 것이 좋습니다.


In [29]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)

module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,order_items,764,5,764,6
2,orders,300,5,300,7
3,products,100,4,100,4


In [30]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)


,dataset,row_duplicate_count,key_column,key_duplicate_count
0,customers,0,customer_id,0
1,products,0,product_id,0
2,orders,0,order_id,0
3,order_items,0,order_item_id,0


In [31]:
module_relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


## 22. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 전처리 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/preprocess_data.py
```

이 스크립트는 전처리 결과 CSV와 요약 보고서를 자동으로 저장합니다.


## 23. LLM과 함께 전처리 코드를 검토하기

LLM에게 전처리 코드를 요청할 때는 실제 고객명, 이메일, 주문 내역을 그대로 입력하지 않는 것이 좋습니다. 컬럼명, 데이터 타입, 결측치 개수, 중복 여부, 처리 목적처럼 구조화된 정보만 제공하세요.

```text
다음 전처리 코드가 안전한지 검토해 주세요.

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지
```


## LLM 답변

| 기존 코드 | 검토 결과 | 이유 |
| --- | --- | ---|
|customers["age"] = customers["age"].fillna(customers["age"].mean())|⚠️ 보완 필요|평균 대체가 항상 적절한 것은 아니며, 결측치가 없거나 데이터가 비정상적인 경우도 확인해야 함|
|customers = customers.drop_duplicates()|⚠️ 주의|모든 컬럼의 값이 같은 행만 제거하며, 고객 ID 중복이나 서로 다른 고객 정보의 충돌은 해결하지 못함|
|orders["order_date"] = pd.to_datetime(orders["order_date"])|⚠️ 보완 필요|잘못된 날짜가 있으면 오류가 발생해 코드가 중단될 수 있으며, 변환 실패 건수를 따로 확인하지 않음|

아래 코드는 원본 보존, 결측치 확인, 날짜 변환 실패 확인, 중복 검증, 숫자형 변환 확인을 포함합니다.

```python
import pandas as pd

# 0. 원본 보존: 전처리용 복사본 생성
customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()


# 1. customers: age 결측치 확인 및 처리
print("[age 결측치]")
print("처리 전 결측치:", customers_clean["age"].isna().sum())

# 평균 대체 전에 실제 숫자형인지 확인
customers_clean["age"] = pd.to_numeric(
    customers_clean["age"], errors="coerce"
)

age_missing = customers_clean["age"].isna().sum()
age_mean = customers_clean["age"].mean()

if age_missing > 0 and pd.notna(age_mean):
    customers_clean["age"] = customers_clean["age"].fillna(age_mean)

print("평균 대체 후 결측치:", customers_clean["age"].isna().sum())


# 2. customers: 중복 확인
print("\n[customers 중복 확인]")
print("전체 행 중복:", customers_clean.duplicated().sum())
print(
    "customer_id 중복:",
    customers_clean["customer_id"].duplicated().sum()
)

# 중복은 확인만 하고, 자동 삭제하지 않음
# customer_id 중복이 있다면 원인을 확인한 후 처리


# 3. orders: 날짜 변환 및 실패 확인
print("\n[order_date 날짜 변환]")

orders_clean["order_date"] = pd.to_datetime(
    orders_clean["order_date"], errors="coerce"
)

date_failed = orders_clean["order_date"].isna().sum()
print("날짜 결측 또는 변환 실패:", date_failed)

if date_failed > 0:
    print("확인할 행:")
    print(
        orders_clean.loc[
            orders_clean["order_date"].isna()
        ].head(10)
    )


# 4. order_items: 숫자형 변환 및 실패 확인
print("\n[수량 및 단가 변환]")

for col in ["quantity", "unit_price"]:
    original = order_items_clean[col]

    converted = pd.to_numeric(original, errors="coerce")

    # 원래 결측치와 변환 실패를 구분
    conversion_failed = (
        original.notna() & converted.isna()
    ).sum()

    print(f"{col} 변환 실패:", conversion_failed)

    order_items_clean[col] = converted

# 결측치 확인
print("quantity 결측치:", order_items_clean["quantity"].isna().sum())
print("unit_price 결측치:", order_items_clean["unit_price"].isna().sum())


# 5. line_total 계산
order_items_clean["line_total"] = (
    order_items_clean["quantity"]
    * order_items_clean["unit_price"]
)

print("\n[line_total 확인]")
print("line_total 결측치:", order_items_clean["line_total"].isna().sum())
print(order_items_clean[
    ["quantity", "unit_price", "line_total"]
].head())
```

## 24. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
2. `customers_clean`에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
3. `orders_clean`에서 월별 주문 건수를 계산하세요.
4. `order_items_clean`에서 `line_total`이 큰 순서대로 상위 10개를 확인하세요.
5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.
6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.


In [36]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
# hint: missing_summary(df).reset_index()를 활용해 보세요.

missing_all = pd.concat(
    [
        missing_summary(df).rename_axis('column').reset_index().assign(dataset=name)
        for name, df in processed_data.items()
    ],
    ignore_index=True,
)[['dataset', 'column', 'missing_count', 'missing_ratio']]

display(missing_all)
print('컬럼 수(행):', len(missing_all))
print('결측이 있는 컬럼 수:', int((missing_all['missing_count'] > 0).sum()))

,dataset,column,missing_count,missing_ratio
0,customers,customer_id,0,0.0
1,customers,name,0,0.0
2,customers,gender,0,0.0
3,customers,age,0,0.0
4,customers,city,0,0.0
5,customers,signup_date,0,0.0
6,products,product_id,0,0.0
7,products,product_name,0,0.0
8,products,category,0,0.0
9,products,price,0,0.0


컬럼 수(행): 23
결측이 있는 컬럼 수: 0


In [37]:
# 과제 2. customers_clean에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.

age_outliers = customers_clean[
    (customers_clean['age'] < 18) | (customers_clean['age'] > 100)
]

print('나이가 18세 미만이거나 100세 초과인 고객 수:', len(age_outliers))
display(age_outliers)

# 후보가 없더라도 실제 범위를 함께 확인합니다.
display(customers_clean['age'].describe())

나이가 18세 미만이거나 100세 초과인 고객 수: 0


,customer_id,name,gender,age,city,signup_date


count    150.000000
mean      42.086667
std       15.613166
min       19.000000
25%       29.000000
50%       40.000000
75%       57.000000
max       69.000000
Name: age, dtype: float64

In [38]:
# 과제 3. orders_clean에서 월별 주문 건수를 계산하세요.

monthly_orders = (
    orders_clean.groupby('order_month')
    .size()
    .reset_index(name='order_count')
    .sort_values('order_month')
)

display(monthly_orders)
print('월별 주문 건수 합계:', monthly_orders['order_count'].sum())
print('orders_clean 행 수:', len(orders_clean))
print('주문 기간:', orders_clean['order_date'].min(), '~', orders_clean['order_date'].max())

,order_month,order_count
0,2025-09,10
1,2025-10,27
2,2025-11,21
3,2025-12,35
4,2026-01,29
5,2026-02,16
6,2026-03,34
7,2026-04,28
8,2026-05,27
9,2026-06,26


월별 주문 건수 합계: 300
orders_clean 행 수: 300
주문 기간: 2025-09-14 00:00:00 ~ 2026-09-13 00:00:00


In [39]:
# 과제 4. order_items_clean에서 line_total이 큰 순서대로 상위 10개를 확인하세요.

top10_line_total = order_items_clean.nlargest(10, 'line_total')

display(top10_line_total)
print('상위 10개 line_total 합계:', top10_line_total['line_total'].sum())

,order_item_id,order_id,product_id,quantity,unit_price,line_total
22,23,10,99,5,200000,1000000
85,86,39,99,5,200000,1000000
183,184,76,99,5,200000,1000000
78,79,35,70,5,198000,990000
493,494,190,70,5,198000,990000
45,46,21,58,5,197000,985000
59,60,25,58,5,197000,985000
113,114,51,58,5,197000,985000
395,396,152,58,5,197000,985000
569,570,221,58,5,197000,985000


상위 10개 line_total 합계: 9905000


In [40]:
# 과제 5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 확인하세요.

row_change = comparison.assign(
    row_diff=comparison['rows_processed'] - comparison['rows_raw'],
    column_diff=comparison['columns_processed'] - comparison['columns_raw'],
)[['dataset', 'rows_raw', 'rows_processed', 'row_diff', 'columns_raw', 'columns_processed', 'column_diff']]

display(row_change)
print('행 수가 줄어든 데이터셋:', row_change.loc[row_change['row_diff'] < 0, 'dataset'].tolist())


,dataset,rows_raw,rows_processed,row_diff,columns_raw,columns_processed,column_diff
0,customers,150,150,0,6,6,0
1,products,100,100,0,4,4,0
2,orders,300,300,0,5,7,2
3,order_items,764,764,0,5,6,1


행 수가 줄어든 데이터셋: []


### 과제 5 설명

위 결과 기준으로 전처리 전후에 행 수가 줄어든 데이터셋은 없다. customers 150행, products 100행, orders 300행, order_items 764행이 처리 전후에 모두 같습니다.

행이 줄어들 수 있는 처리는 완전 중복 행 제거(셀 9)와 0 이하 가격,수량,단가 제외(셀 14)였다. 이번 데이터에서는 전체 행 중복이 0건, `price`, `quantity`, `unit_price`가 0 이하인 행이 0건이어서 제거된 행이 없다. 결측치는 삭제하지 않고 대체하는 기준(셀 8)이라 행 수에 영향을 주지 않는다.

반면 orders에는 파생 컬럼 `order_month`, `order_dayofweek`가 추가되어 5열에서 7열로, order_items에는 `line_total`이 추가되어 5열에서 6열로 늘었다.

행 수가 같다는 결과는 "전처리가 데이터를 개선했다"는 뜻이 아니라, 이번 샘플 데이터에는 제거할 만한 행이 없었다는 뜻이다. 다른 데이터에서는 같은 기준으로 행이 줄어들 수 있으므로, 줄어든 행이 있다면 어떤 기준 때문인지 설명할 수 있어야 한다.

### 과제 6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트

실제 고객명, 이메일, 주문 내역은 넣지 않고 구조와 요약 정보만 제공한다. 아래 수치는 이 노트북의 출력(셀 7, 셀 8)에서 가져온 것이다.

```text
customers 데이터 품질 요약
- rows: 150, columns: 6 (customer_id, name, gender, age, city, signup_date)
- customer_id missing/duplicate: 0 / 0
- age: 숫자형, missing 0건 (중앙값 40.0)
- city: 문자열(범주형), missing 0건
- 현재 샘플 데이터에는 결측이 없지만, 실제 업무 데이터에서 결측이 생겼을 때의 기준을 정하고 싶습니다.

현재 적용한 처리 기준
- age 결측: 중앙값으로 대체
- city 결측: "Unknown"으로 대체

다음을 검토해 주세요.
1. age 결측치 처리 후보(삭제, 평균 대체, 중앙값 대체, 결측 유지 + 결측 여부 표시 컬럼 추가 등)의 장단점을 비교해 주세요.
2. city 결측을 "Unknown"으로 대체했을 때 분석 결과에 어떤 영향을 줄 수 있는지 설명해 주세요.
3. 각 방법을 적용하기 전후에 확인해야 할 검증 항목(대체한 건수, 분포 변화, 행 수 변화 등)을 제안해 주세요.
4. 결측 처리를 하면 안 되는 컬럼(고유 ID, 핵심 날짜, 금액, 상태값 등)이 있다면 이유와 함께 알려 주세요.

주의: 실제 업무 의미를 확인하지 않은 값은 단정하지 말고, 추가로 확인해야 할 질문을 알려 주세요.
```

#### LLM 답변을 받은 뒤 사람이 확인할 것

- 원본(raw)을 덮어쓰는 코드가 있는가?
- 변환 실패 건수를 확인하는가?
- 결측 처리의 근거가 설명되어 있는가?
- 키 중복이나 이상값을 자동으로 삭제하지 않는가?
- 처리 후 PK/FK 관계를 다시 검증하는가?
- 변경 내역(몇 건이 바뀌었는지)을 기록하는가?

## 25. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 원본 데이터와 전처리 데이터 분리
- 결측치 개수와 비율 확인
- 나이 결측치 중앙값 대체, 도시 결측치 Unknown 처리
- 중복 행과 주요 ID 중복 점검
- 문자열 앞뒤 공백 제거와 상태값 표기 통일
- 날짜형 변환과 주문 월/요일 파생 컬럼 생성
- 숫자형 변환과 변환 실패 건수 확인
- 이상값 후보 확인과 실습용 처리 기준 적용
- 주문 상세 금액 `line_total` 생성
- 파일 간 키 관계 재확인
- 전처리 결과 CSV와 요약 보고서 저장
- 전처리 함수를 `src/preprocessing.py` 소스 모듈로 분리

다음 장에서는 전처리된 데이터를 바탕으로 탐색적 데이터 분석, 즉 EDA를 수행합니다.


# Chapter 05 제출 답안 양식. 분석을 믿을 수 있게 만드는 데이터 전처리

> 주 제출물은 실행 완료 Notebook `chapter05/chapter05.ipynb`입니다. 이 양식의 항목을 Notebook의 Markdown 셀로 추가해 작성합니다.

## 0. 제출 정보
- 이름: 배성윤
- GitHub ID: aeuyui
- 작성일: 2026-10-12
- 최종 제출 URL: https://github.com/aeuyui/llm-data-analysis-study/blob/main/chapter05/chapter05.ipynb

## 1. 전처리 전 상태 기록
- 각 데이터 shape:

|data file|shape|column|
|---|---|---|
customers|(150,6)|'customer_id', 'name', 'gender', 'age', 'city', 'signup_date'|
products|(100,4)|'product_id', 'product_name', 'category', 'price'|
orders|(300,5)|'order_id', 'customer_id', 'order_date', 'payment_method', 'order_status'|
order_items|(764,5)|'order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price'|

- 결측치: 4개 데이터 파일 모두 결측치 없음
- 전체 중복: 4개 데이터 파일 모두 중복 없음
- 주요 ID 중복: 4개 데이터 파일 모두 ID 중복 없음
- 타입 문제 후보: 문자열 데이터는 앞뒤 공백이 존재할 수 있으므로, 실제로 같은 값을 의미하지만 pandas에서는 다른 값으로 집계될 수 있다.

![전처리 전 상태](images/step01_before(1).png)

![전처리 전 상태](images/step01_before(2).png)

### 결과 관찰
- 4개 파일 모두 결측 0건, 전체 중복 0건, 주요 ID 중복 0건이다.
- orders는 300행, order_items는 764행이다. 주문 1건당 여러 개의 상세 행이 연결되는 구조이므로, order_items.order_id가 반복되는 것은 한 주문에 여러 상품이 들어갈 수 있는 자연스러운 반복이다.

### 나의 해석과 판단
결측, 전체 중복, ID 중복은 모두 0건이라 이 세 항목에서는 처리할 문제가 없다.
아직 확인하지 않은 것은 문자열 공백, 숫자/날짜 타입, 범주 표기, 이상값, 파일 간 관계이므로 이 중에서 같은 값을 다르게 집계할 수 있는 문자열 공백과 계산 및 월별 분석에 영향을 주는 숫자/날짜 타입 변환을 먼저 확인하는 것이 순서라고 판단했다. 이후 범주값 표기, 이상값 후보를 보고, 마지막에 파일 간 관계를 다시 검증할 것이다.

### 업무·분석적 의미
이 표가 전처리 전/후 비교의 기준점이다. 이후 행 수나 열 수가 달라지면 이 값과 비교해 의도한 변화인지 판단할 수 있다. 또 주문 건수(300)와 주문 상세 행 수(764)는 집계 단위가 다르므로 분석할 때 구분해야 한다.

### 한계와 추가 확인 사항
- step1에서는 결측, 전체 중복, ID 중복만 확인했다. 공백, 타입, 범주 표기, 값의 범위, 파일 간 관계는 2~4단계에서 확인한다.
- 기본 샘플 데이터라서 문제가 적게 나온 것일 수 있다. 실제 업무 데이터에서도 같은 결과가 나온다고 일반화할 수 없다.

## 2. 문자열·타입·결측 처리
- 적용한 문자열 정리: 4개 데이터의 문자열 컬럼 앞뒤 공백을 제거했다. 결측값은 문자열 'nan'으로 바뀌지 않도록 결측이 아닌 값에만 적용했다. 빈 문자열을 결측으로 바꾸는 처리는 적용하지 않았다.
- 숫자 변환 대상과 실패 건수: price, quantity, unit_price를 쉼표 제거 후 숫자로 변환했다.
- 날짜 변환 대상과 실패 건수: order_date, signup_date를 날짜형으로 변환했다. 주문일 범위는 2025-09-14 ~ 2026-09-13이다.
- 결측 처리 기준: age는 중앙값(40.0)으로 대체, city는 'Unknown'으로 대체하는 기준을 적용했다. 원본 결측이 0건이었으므로 실제로 바뀐 값은 없다.

![타입과 결측 처리](images/step02_cleaning.png)

### 결과 관찰
- 숫자 변환 3개 컬럼과 날짜 변환 2개 컬럼 모두 실패가 0건이다.
- 가장 늦은 주문일(2026-09-13)이 작성일(2026-10-12)보다 이전이라 미래 날짜는 없다.
- 공백 제거 후 order_status는 completed, cancelled, refunded 3개 값만 남았다.
- age와 city는 결측이 없어서 중앙값/Unknown 대체로 바뀐 값이 없다.

### 나의 해석과 판단
- 공백 제거는 'Seoul'과 ' Seoul'이 서로 다른 값으로 집계되는 문제를 막기 위해 적용했다. 결측은 건드리지 않아 정보를 잃지 않는다.
- errors="coerce"는 변환 실패를 NaN/NaT로 바꾸므로, 실행만 하지 않고 실패 건수를 직접 확인해야 한다. 이번에는 모두 0건이라 그대로 사용할 수 있었다.
- age는 숫자형이라 중앙값을 쓰고, city는 범주형이라 대표값이 없으므로 Unknown으로 대치하기로 하였다. 
- 삭제나 대체는 정보 손실을 만들 수 있다. 중앙값 대체는 여러 고객이 같은 나이로 보이게 하여 나이 분포를 좁히고, 왜 비어 있었는지라는 정보가 사라진다. Unknown은 결측을 하나의 범주처럼 취급하게 만든다. 고유 ID, 핵심 날짜, 금액, 상태값은 근거 없이 대표값으로 대체하지 않는다.

### 업무·분석적 의미
날짜가 날짜형으로 바뀌어 월별/요일별 분석이 가능하고, 가격과 수량이 숫자형이라 이후 금액 계산(line_total)에 사용할 수 있다. 공백을 정리해 두었으므로 도시·상태별 집계가 쪼개지지 않는다.

### 한계와 추가 확인 사항
- 실행파일에서는 공백 제거로 몇 건이 바뀌었는지는 출력하지 않아 원본에 공백이 실제로 있었는지는 확인하지 못했다.
- age는 셀 8에서 숫자 변환 직후 바로 결측을 채우므로, 변환 실패가 있었더라도 중앙값에 가려질 수 있다. age의 변환 실패 건수는 따로 출력되지 않았다.

## 3. 중복·범주 표준화·이상값 후보
- 제거/보류한 중복: 전체 행 중복은 4개 데이터 모두 0건이라 제거된 행이 없다. 고유 ID 중복도 0건이라 보류할 것이 없다. order_items.order_id의 반복은 한 주문에 여러 상품이 들어갈 수 있는 자연스러운 반복이므로 중복으로 보지 않았다.
- 표준화한 범주값: order_status에 complete/완료  completed, cancel/취소 -> cancelled, refund/환불 -> refunded 매핑을 적용했다. 표준화 전후 값 분포가 같아서(completed 184, cancelled 64, refunded 52) 실제로 바뀐 값은 없다.
- 허용값 밖 값: order_status는 대표 상태 3개(completed, cancelled, refunded) 외의 값이 0건이다. 다른 범주 컬럼(gender, payment_method, category 등)은 점검하지 않았다.
- 이상값 후보: price <= 0, quantity <= 0, unit_price <= 0 모두 0건이다. 값 범위는 price 5,000~200,000, quantity 1~5, unit_price 5,000~200,000이다.

![중복 범주 이상값](images/step03_rules(1).png)

![중복 범주 이상값](images/step03_rules(2).png)

### 결과 관찰
- 중복과 ID 중복은 모두 0건이라 삭제된 행이 없다.
- 상태값은 표준화 전에도 이미 3개 대표값뿐이었다.
- 0 이하 가격·수량·단가가 없어서 이상값 처리로 제외된 행이 없다. 처리 전후 shape가 같다.

### 나의 해석과 판단
- 이상값은 일반적인 범위를 벗어난 값일 뿐 항상 오류는 아니다. 이상적인 범위에는 벗어나지만 유의미한 데이터일 수 있으므로, 섣불리 삭제하는 것이 아니라 다른 변수들까지 종합하여 의미를 파악하고 삭제 또는 유지를 결정해야한다.
- 삭제하면 파일 간 관계도 깨질 수 있다. products에서 일부 상품을 제외하면 order_items에 그 product_id가 남아 참조가 끊긴다.
- 이번 데이터는 후보가 0건이라 제외 없이 그대로 두는 것이 맞다고 판단했다.

### 업무·분석적 의미
중복,상태값,이상값 점검 결과가 모두 깨끗하므로 행 수(300 주문, 764 주문 상세)를 그대로 신뢰하고 이후 분석에 쓸 수 있다. 상태값이 3개로 통일되어 있어 주문 상태별 집계가 쪼개지지 않는다.

### 한계와 추가 확인 사항
- 노트북에서는 중복 제거가 문자열 공백 제거보다 먼저 실행된다. 공백만 다른 중복 행이 있었다면 걸러지지 않았을 수 있다. 이번 데이터에서는 제거된 행이 0건이라 확인된 문제는 없다.
- order_status 외의 범주 컬럼은 허용값을 점검하지 않았다. city는 셀 10에서 상위 10개 값만 출력했다.

## 4. PK/FK와 파생 컬럼 재검증
- 없는 `customer_id`: 0건 (orders.customer_id가 customers.customer_id에 모두 존재)
- 없는 `order_id`: 0건 (order_items.order_id가 orders.order_id에 모두 존재)
- 없는 `product_id`: 0건 (order_items.product_id가 products.product_id에 모두 존재)
- 생성한 파생 컬럼: order_month(주문 월, 예: 2026-07), order_dayofweek(주문 요일, 예: Monday), line_total(주문 상세 한 행의 수량 x 단가)
- `line_total` 검증: 상위 5행을 직접 곱해 값이 일치함을 확인했다(예: 3 x 102,000 = 306,000). quantity와 unit_price는 숫자 변환 실패가 0건이고 0 이하 값이 없어 곱셈 결과가 안전하다. 전체 합계는 255,610,000이다.

![관계와 파생 컬럼 검증](images/step04_validation.png)

### 결과 관찰
- 세 가지 참조 관계 모두 깨진 행이 0건이다. 직접 계산한 결과와 src.preprocessing의 validate_relationships 결과가 같다.
- 파생 컬럼 3개가 만들어져 orders는 5->7열, order_items는 5->6열이 되었다. 행 수는 그대로다(300, 764).
- line_total 합계는 255,610,000이다.

### 나의 해석과 판단
- 전처리 중 행을 삭제하면 관계가 깨질 수 있다. 예를 들어 products에서 일부 상품을 제외했는데 order_items에는 그 상품 ID가 남을 수 있다. 그래서 전처리가 끝난 뒤 다시 확인해야 한다. 이번에는 제외된 행이 없어서 0건이었다.
- line_total은 주문 상세 한 행의 수량 x 단가이다. 합계 255,610,000을 곧바로 매출이나 순매출이라고 할 수는 없다. 이 합계는 order_items 전체를 더한 값이어서 주문 상태를 구분하지 않았고, orders에는 cancelled 64건과 refunded 52건이 있으므로 이 주문들의 상세 금액도 포함되어 있다.
- 이 검증은 order_items에서 부모 파일로 가는 방향만 본다. 주문이 없는 고객이나 상세가 없는 주문이 있는지는 확인하지 않았다.

## 5. 전처리 전/후 비교
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | 150 / 100 / 300 / 764 (합계 1,314) | 150 / 100 / 300 / 764 (합계 1,314) | 중복 0건, 0 이하 값 0건이라 제거된 행이 없다. 결측은 삭제하지 않고 대체하는 기준이라 행이 줄지 않는다. |
| 결측 | 0 (4개 데이터 모두) | 0 (customers 전체와 변환 대상 컬럼 기준) | 원본 결측이 없어 대체된 값이 없다. 변환으로 새로 생긴 NaN/NaT도 없다. |
| 중복 | 0 (전체 행, 주요 ID) | 0 | 제거할 중복이 없었다. |
| 변환 실패 | 변환 전이라 해당 없음 | 0 (price, quantity, unit_price, order_date, signup_date) | errors="coerce"로 변환한 뒤 실패 건수를 직접 확인했다. |

![전처리 전후 비교](images/step05_before_after.png)

### 결과 관찰
- 4개 데이터 모두 행 수가 전처리 전후로 같다(합계 1,314).
- 열 수는 orders 5->7(order_month, order_dayofweek), order_items 5->6(line_total)로 늘었다. 나머지는 같다.
- order_status 분포는 표준화 전후 모두 completed 184 / cancelled 64 / refunded 52로 같다.
- 노트북 직접 구현과 src.preprocessing 모듈의 비교 결과가 일치한다.

### 나의 해석과 판단
- 행이 줄어들 이유가 되는 조건이 모두 0건(전체 중복 0, 변환 실패 0, 0 이하 가격/수량/단가 0)이었기 때문에 행 수가 같은 것을 의도한 결과라고 판단했다. 행이 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 한다.
- 열 수 증가는 새 파생 컬럼 3개로 설명된다.
- 값 분포(order_status)도 바뀌지 않아 표준화가 값을 의도치 않게 바꾸지 않았다고 판단했다.

### 업무·분석적 의미
전처리로 데이터가 사라지지 않았으므로, 이후 분석에서 주문 300건과 주문 상세 764행을 그대로 기준으로 쓸 수 있다. 다만 order_date 등이 날짜형/숫자형으로 바뀌었고 파생 컬럼이 늘어, 월별/요일별 분석과 금액 계산이 가능해졌다.

### 한계와 추가 확인 사항
- 원본 데이터의 품질이 이미 좋았기 때문에, 전/후가 같다는 결과는 전처리가 검증을 통과했다는 뜻이지, 전처리가 데이터를 개선했다고는 보기 어렵다.
- 처리 후 결측은 customers 전체와 변환 대상 컬럼만 확인했다. 4개 데이터 전체의 처리 후 결측은 따로 출력하지 않았다.
- 전/후 비교는 shape 위주이고, order_status 외 컬럼(age, price 등)의 분포 전후 비교는 하지 않았다. 처리 전 상태의 값 통계(describe)는 출력하지 않았고, 셀 14의 통계는 변환 이후 값이다.

## 6. 재현 가능한 전처리 확인
- 생성된 clean CSV: data/processed/customers_clean.csv, products_clean.csv, orders_clean.csv, order_items_clean.csv (4개). 스크립트 실행 결과에 저장 경로 4개가 출력되었다.
- 생성된 요약 보고서: reports/ch05_preprocessing_summary.md (스크립트 출력에 경로가 표시됨)
- `python scripts/preprocess_data.py` 재실행 결과: xx최종 PASS (처음 1회 FAIL 후 수정)xx
  - 처음 실행에서는 `ModuleNotFoundError: No module named 'src'`(스크립트 22행 `from src.data_loader import load_sales_data`)로 실패했다.
  - 원인은 `python scripts/preprocess_data.py`로 실행하면 scripts 폴더만 import 검색 경로에 들어가서, 프로젝트 루트의 src 폴더를 찾지 못한 것이다. 노트북은 셀 3에서 프로젝트 루트를 직접 sys.path에 추가했기 때문에 같은 import가 성공했다.
  - 스크립트의 import 앞에 아래 코드를 추가해 수정했다.

```python
import sys
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
```

  - 수정 후 재실행은 오류 없이 종료되었고, 전처리 전후 크기(150x6->150x6, 100x4->100x4, 300x5->300x7, 764x5->764x6), 중복 점검(행,ID 중복 모두 0), 관계 점검(invalid_count 모두 0), 저장 파일 4개, 보고서 경로가 출력되었다.
- 재실행 후 달라진 점: 수정 전에는 import 단계에서 실패했고, 수정 후에는 끝까지 실행되었다. 전처리 전후 크기,중복,관계 결과는 노트북 셀 17, 셀 21 결과와 같아서 노트북과 스크립트 결과가 일치한다. 노트북이 저장한 clean CSV와 보고서와 같은 경로에 저장했다.

![재실행 결과](images/step06_reproducible(1).png)

![재실행 결과](images/step06_reproducible(2).png)

### 나의 해석과 판단
- 노트북에서 실행이 성공하는 것은 셀을 위에서부터 순서대로 실행했고, 앞 셀의 변수와 경로 설정(셀 3)이 남아 있어서 성공한 것이다. 노트북에서는 같은 import가 되었지만, 스크립트는 경로 설정이 없어서 실패했다.
- 스크립트는 같은 입력(data/raw)에서 같은 순서로 처음부터 끝까지 실행되고, 실행할 때마다 결과 파일과 보고서를 만든다. 데이터를 다시 받거나 기준을 바꿔도 같은 명령 하나로 반복할 수 있고, 다른 사람도 재현할 수 있다.
- 노트북 직접 구현(셀 17)과 src 모듈/스크립트 결과가 같다는 점이 재현성을 뒷받침한다. 오류가 났을 때 처음에 실패한 사실과 수정 내용을 함께 기록해 두는 것도 재현을 위한 정보라고 판단했다.
- 스크립트가 쓰는 대상은 data/processed와 reports이고 data/raw는 읽기만 하므로 원본은 보존된다(스크립트 코드 기준).

## 7. Chapter 05 최종 판단
### 내가 정한 전처리 원칙 3가지
1. 원본(data/raw)은 덮어쓰지 않고 복사본에서 처리하며, 결과는 data/processed에 따로 저장한다.
2. 값을 바꾸거나 제외하기 전에 기준과 영향 건수(변환 실패, 대체, 제외)를 확인하고, 처리 전후 비교를 남긴다.
3. 이상값과 결측은 자동으로 삭제하거나 대체하지 않고 후보로 보고 업무 의미를 확인하며, 처리 후에는 PK/FK 관계를 다시 검증한다.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유
0 이하의 price, quantity, unit_price를 기준만으로 자동 제외하는 처리(노트북 셀 14)가 가장 위험하다고 판단했다. 단순히 값만 보고 오류로 단정해 지우면 정상적인 업무 데이터를 잃을 수 있다.
- products에서 상품을 제외하면 order_items에 그 product_id가 남아 관계가 깨질 수 있고, 제외한 행만큼 line_total 합계와 주문 건수도 달라진다.
- 이번 데이터는 해당 값이 0건이라 실제로 제외된 행이 없었다. 그러나 코드는 조건에 맞으면 자동으로 지우도록 되어 있어, 다른 데이터에 그대로 쓰면 조용히 행이 사라질 수 있고, 다른 데이터에도 영향을 미쳐 분석에 지장이 생길 수 있다.

### 다음 EDA에서 특히 주의할 데이터 특성
- 주문 상태: completed 184 / cancelled 64 / refunded 52이다. line_total 합계(255,610,000)는 상태를 구분하지 않은 값이므로 매출로 부르지 않고, 상태별로 나눠 계산해야 한다.
- 집계 단위: 주문(300건)과 주문 상세(764행)는 단위가 다르다. 둘을 합치면 주문 수준 지표가 중복 집계될 수 있으니 어느 단위로 세는지 명시해야 한다.
- 기간: 주문일은 2025-09-14 ~ 2026-09-13이다. 첫 달(2025-09)과 마지막 달(2026-09)은 한 달 전체가 아닐 수 있어 월별 비교에 주의해야 한다.
- 샘플 데이터라서 결측·이상값이 없었고, 실제 업무 데이터의 분포와 같다고 볼 수 없다.

### 현재 전처리의 한계
- 원본이 이미 깨끗해서 결측·중복·이상값 처리 규칙이 실제로 작동하는지는 확인하지 못했다. 처리 전후 값이 같다는 것은 검증을 통과했다는 뜻이지 개선했다는 뜻이 아니다.
- 원본의 공백 건수와 처리 전 dtype, order_status 외 범주 컬럼의 허용값은 점검하지 않았다.
- 처리 후 결측은 customers와 변환 대상 컬럼만 확인했고, 관계 검증은 order_items에서 부모 파일 방향만 확인했다.
- 노트북 셀 14의 이상값 제외 기준은 실습용이며 업무 확인을 거치지 않았다.
- 저장한 CSV를 다시 읽어 구조를 확인하는 검증과 보고서 내용 검토는 하지 않았다.

## 최종 제출 체크
- [x] raw 원본을 덮어쓰지 않았습니다.
- [x] 처리 전/후 비교를 남겼습니다.
- [x] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [x] clean CSV와 재실행 결과를 확인했습니다.
- [x] 개인정보/Secret이 없습니다.
- [x] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [x] 최종 Notebook 파일 URL을 제출합니다.